- torch.multiprocessing.spawn 是 PyTorch 提供的<font color='red'>用于启动多个子进程并行执行任务的核心工具。</font>
- 在深度学习领域，<font color='red'>它最主要的用途就是单机多卡分布式训练</font>
  - <font color='red'>即在一台机器上启动 N 个进程，每个进程控制一张 GPU 进行并行计算。</font>
- <font color='red'>它是对 Python 标准库 multiprocessing 的深度封装和优化，专门解决了多进程与 CUDA（GPU）交互时的各种兼容性和安全性问题。</font>

## 🛠️ 核心参数解析
torch.multiprocessing.spawn 的函数签名如下：

- torch.multiprocessing.spawn(fn, args=(), nprocs=1, join=True, daemon=False, start_method='spawn')
- fn (function)：入口函数。<font color='red'>这是每个子进程启动后要执行的目标函数</font>。<font color='green'>它必须定义在模块的顶层（不能是类方法或嵌套函数），因为多进程需要通过 pickle 序列化来传递它。</font>
- args (tuple)：传递给入口函数 fn 的额外参数（以元组形式）。
- nprocs (int)：启动的进程数量。在单机多卡训练中，这通常等于你机器上的 GPU 数量（torch.cuda.device_count()）。
- join (bool)：<font color='red'>如果设为 True（默认），主进程会阻塞等待所有子进程执行完毕后再退出。</font>

## 💡 核心机制：自动注入的 rank
spawn 最巧妙的地方在于它对入口函数 fn 的<font color='red'>调用方式</font>。
- <font color='red'>它会自动将进程的索引（index）作为第一个参数传递给 fn。</font>

也就是说，如果你调用 mp.spawn(my_func, args=('hello',), nprocs=4)，PyTorch 实际上会在后台依次执行：
- my_func(0, 'hello') (在进程 0 中)
- my_func(1, 'hello') (在进程 1 中)
- my_func(2, 'hello') (在进程 2 中)
- my_func(3, 'hello') (在进程 3 中)
<font color='red'>这个自动注入的索引（通常被称为 rank 或 local_rank）</font>，<font color='blue'>就是我们在分布式训练中区分“我是几号进程”、“我该控制哪张显卡”的核心依据。</font>

## 💻 基础入门示例
下面是一个最基础的示例，展示如何启动 4 个进程并打印各自的编号：

In [ ]:
import torch.multiprocessing as mp

def worker(rank, world_size):
    """入口函数，rank 是由 spawn 自动注入的进程编号"""
    print(f"你好！我是 {rank} 号进程，当前总共有 {world_size} 个进程在并行工作。")

if __name__ == "__main__":
    world_size = 4  # 启动 4 个进程
    # 启动多进程，args 中只需要传 world_size，rank 会自动传给 worker 的第一个参数
    mp.spawn(worker, args=(world_size,), nprocs=world_size, join=True)

## 🚀 进阶实战：结合分布式训练 (DDP)
在实际的单机多卡训练中，spawn 通常与 torch.distributed 配合使用。
每个进程启动后，会利用自己的 rank 去绑定对应的 GPU，并初始化分布式环境。

In [ ]:
import os
import torch
import torch.distributed as dist
import torch.multiprocessing as mp
import torch.nn as nn

def setup(rank, world_size):
    """初始化分布式环境"""
    os.environ['MASTER_ADDR'] = 'localhost'
    os.environ['MASTER_PORT'] = '12355'
    # 初始化进程组，所有进程通过 rank 互相识别
    dist.init_process_group("gloo", rank=rank, world_size=world_size)

def cleanup():
    """清理分布式环境"""
    dist.destroy_process_group()

def train_process(rank, world_size):
    """每个子进程执行的训练逻辑"""
    print(f"进程 {rank} 启动，正在绑定 GPU: {rank}")
    
    # 1. 初始化分布式环境
    setup(rank, world_size)
    
    # 2. 绑定当前进程到指定的 GPU (rank 0 -> cuda:0, rank 1 -> cuda:1 ...)
    # 注意：如果没有真实 GPU，这里可以用 cpu 模拟，但真实场景是 cuda:rank
    device = torch.device(f"cuda:{rank}" if torch.cuda.is_available() else "cpu")
    torch.cuda.set_device(device)
    
    # 3. 创建模型并移动到当前进程的 GPU 上
    model = nn.Linear(10, 1).to(device)
    # 实际场景中这里会用 DistributedDataParallel(model, device_ids=[rank]) 包装
    
    # 4. 模拟训练
    print(f"进程 {rank} 在设备 {device} 上开始训练...")
    
    # 5. 训练结束，清理资源
    cleanup()

if __name__ == "__main__":
    # 获取机器上的 GPU 数量，如果没有 GPU 则模拟 2 个进程
    world_size = torch.cuda.device_count() if torch.cuda.is_available() else 2
    print(f"准备启动 {world_size} 个进程进行并行训练...")
    
    # 使用 spawn 启动多进程
    mp.spawn(train_process, args=(world_size,), nprocs=world_size, join=True)

## ⚠️ 为什么推荐使用 spawn 而不是 fork？
Python 的多进程有 fork、spawn、forkserver 三种启动方式。torch.multiprocessing 默认且强烈推荐 spawn，原因如下：
- 安全性：fork 会直接复制父进程的内存状态。如果父进程中已经有后台线程（比如 PyTorch 的数据加载器或 CUDA 驱动线程），fork 出来的子进程极易发生死锁或崩溃。
- CUDA 兼容性：CUDA 运行时不支持 fork。在涉及 GPU 计算时，使用 fork 会直接报错。而 spawn 会启动一个全新的、干净的 Python 解释器进程，完全避免了这些问题，是多 GPU 场景下最安全的选择。